In [1]:
RANDOM_STATE= 42
BATCH_SIZE = 128
LR = 2e-4
EPOCHS = 5

# xtransformer Config
HIDDEN_DIM=300
DEPTH=4
HEADS=4
MAX_LEN=256

MAX_VOCAB=20000
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
PAD_ID = 0
UNK_ID = 1

In [2]:
!pip install x-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.8/97.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.0/103.0 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.2 MB/s eta 0:00:00


In [ ]:
import re
import nltk
import random
import numpy as np
import pandas as pd
# import seaborn as sns
# import matplotlib.pyplot as plt

try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')
import torch.nn as nn
from sklearn.metrics import f1_score
from gensim.models import Word2Vec
from collections import Counter
from x_transformers import TransformerWrapper, Decoder
import inspect
print(inspect.signature(Decoder.__init__))

# Setting as large the xtick and ytick font sizes in graphs

# plt.rcParams['xtick.labelsize'] = 'large'
# plt.rcParams['ytick.labelsize'] = 'large'

/usr/local/lib/python3.12/dist-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils


(self, **kwargs)


<a id='IMDB'></a>
# IMDB dataset
We retrieve from [Kaggle](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews?resource=download) the csv file "IMDB Dataset.csv" consisting of 50'000 IMDB movies and TV shows reviews with their positive or negative sentiment classification.

In [4]:
# Storing the csv file into a DataFrame "df"

df = pd.read_csv('../input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv')
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [5]:
df.sentiment = [1 if s == 'positive' else 0 for s in df.sentiment]
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1
...,...,...
49995,I thought this movie did a down right good job...,1
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",0
49997,I am a Catholic taught in parochial elementary...,0
49998,I'm going to have to disagree with the previou...,0


<a id='preprocessing'></a>
# Data preprocessing
First, we use regular expressions to make the following transformations to the reviews:

- remove punctuation marks
- remove HTML tags
- remove URL's
- remove characters which are not letters or digits
- remove successive whitespaces
- convert the text to lower case
- strip whitespaces from the beginning and the end of the reviews

In [6]:
# Storing in "before_process" a random example of review before preprocessing
# Defining and applying the function "process" performing the transformations of the reviews
# Storing in "after_process" the example of review after preprocessing

idx = random.randint(0, len(df)-1)
before_process = df.iloc[idx][0]

def process(x):
    x = re.sub('[,\.!?:()"]', '', x)
    x = re.sub('<.*?>', ' ', x)
    x = re.sub('http\S+', ' ', x)
    x = re.sub('[^a-zA-Z0-9]', ' ', x)
    x = re.sub('\s+', ' ', x)
    return x.lower().strip()

df['review'] = df['review'].apply(lambda x: process(x))
after_process = df.iloc[idx][0]
after_process

<>:9: SyntaxWarning: invalid escape sequence '\.'
<>:11: SyntaxWarning: invalid escape sequence '\S'
<>:13: SyntaxWarning: invalid escape sequence '\s'
<>:9: SyntaxWarning: invalid escape sequence '\.'
<>:11: SyntaxWarning: invalid escape sequence '\S'
<>:13: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_55/4004618312.py:9: SyntaxWarning: invalid escape sequence '\.'
  x = re.sub('[,\.!?:()"]', '', x)
/tmp/ipykernel_55/4004618312.py:11: SyntaxWarning: invalid escape sequence '\S'
  x = re.sub('http\S+', ' ', x)
/tmp/ipykernel_55/4004618312.py:13: SyntaxWarning: invalid escape sequence '\s'
  x = re.sub('\s+', ' ', x)
/tmp/ipykernel_55/4004618312.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  before_process = df.iloc[idx][0]
/tmp/ipykernel_55/4004618312.py:17: FutureWarning: Series.__

'why couldn t the end of the movie have been sean connery s men fighting the french instead of the germans ever since the french had occupied algeria in 1830 the tribes from morocco and those of algeria were making raids on the french military and civilian settlements this movie could have been a continuous of that historical aspect where the french had seize the rasuadli so his followers would not be raiding algeria and then his followers would have attacked the french to free him the movie is still stereotypical of shootouts between the germans and the americans when the americans shoot the germans their guns even the pistols make loud noises create large bloody bullet wounds and their enemies are screaming after being shot when germans shoot at the americans their guns don t make large sounds do not create bloody wounds and their enemies make little or no sound after being shot in real life the american krag rifle was the worst rifle america had ever produce until the early version 

Next, we remove stopwords from the reviews using the [word_tokenize()](https://www.nltk.org/_modules/nltk/tokenize.html#word_tokenize) function from the [nltk.tokenize]((https://www.nltk.org/api/nltk.tokenize.html) package.

In [7]:
# Storing in "sw_set" the set of English stopwords provided by nltk
# Defining and applying the function "sw_remove" which remove stopwords from reviews
# Storing in "after_removal" the example of review after removal of the stopwords

sw_set = set(nltk.corpus.stopwords.words('english'))

def tokenize(text):
    return nltk.tokenize.word_tokenize(text)
    
def sw_remove(x):
    words = tokenize(x.lower())
    filtered_list = [word for word in words if word not in sw_set]
    return filtered_list

df['review'] = df['review'].apply(lambda x: sw_remove(x))
after_removal = sw_remove(after_process)
after_removal

['end',
 'movie',
 'sean',
 'connery',
 'men',
 'fighting',
 'french',
 'instead',
 'germans',
 'ever',
 'since',
 'french',
 'occupied',
 'algeria',
 '1830',
 'tribes',
 'morocco',
 'algeria',
 'making',
 'raids',
 'french',
 'military',
 'civilian',
 'settlements',
 'movie',
 'could',
 'continuous',
 'historical',
 'aspect',
 'french',
 'seize',
 'rasuadli',
 'followers',
 'would',
 'raiding',
 'algeria',
 'followers',
 'would',
 'attacked',
 'french',
 'free',
 'movie',
 'still',
 'stereotypical',
 'shootouts',
 'germans',
 'americans',
 'americans',
 'shoot',
 'germans',
 'guns',
 'even',
 'pistols',
 'make',
 'loud',
 'noises',
 'create',
 'large',
 'bloody',
 'bullet',
 'wounds',
 'enemies',
 'screaming',
 'shot',
 'germans',
 'shoot',
 'americans',
 'guns',
 'make',
 'large',
 'sounds',
 'create',
 'bloody',
 'wounds',
 'enemies',
 'make',
 'little',
 'sound',
 'shot',
 'real',
 'life',
 'american',
 'krag',
 'rifle',
 'worst',
 'rifle',
 'america',
 'ever',
 'produce',
 'early'

<a id='splitting'></a>
# Data splitting and tokenization
We start by splitting our DataFrame into a training and test lists. We use the [train_test_split()](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) function from the [sklearn.model_selection](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.model_selection) module which allow to perform the splitting randomly with respect to the index of the DataFrame.

In [8]:
from sklearn.model_selection import train_test_split

# df tokenized success with nltk
train_rev, tmp_rev, train_sent, tmp_sent = train_test_split(df['review'], df['sentiment'], test_size=0.1, random_state=RANDOM_STATE)
test_rev, val_rev, test_sent, val_sent = train_test_split(tmp_rev, tmp_sent, test_size=0.5, random_state=RANDOM_STATE)


print('\033[1m' + 'train_rev.shape:' + '\033[0m', train_rev)
print('\033[1m' + 'train_rev.shape:' + '\033[0m', train_rev.shape)
print('\033[1m' + 'test_rev.shape:' + '\033[0m', test_rev.shape)
print('\033[1m' + 'val_sent.shape:' + '\033[0m', val_sent.shape)
print('\033[1m' + 'val_sent.shape:' + '\033[0m', val_sent)

train_rev.shape: 40877    [recently, started, watching, show, say, reall...
18057    [return, jedi, often, remembered, wrong, rathe...
19066    [remember, loved, movie, came, 12, years, old,...
20525    [know, last, reviewer, talking, show, pure, en...
5847     [beginning, excited, see, movie, poster, possi...
                               ...                        
11284    [shadow, magic, recaptures, joy, amazement, fi...
44732    [found, movie, quite, enjoyable, fairly, enter...
38158    [avoid, one, terrible, movie, exciting, pointl...
860      [production, quite, surprise, absolutely, love...
15795    [decent, movie, although, little, bit, short, ...
Name: review, Length: 45000, dtype: object
train_rev.shape: (45000,)
test_rev.shape: (2500,)
val_sent.shape: (2500,)
val_sent.shape: 34234    1
28241    0
1226     1
27004    0
35839    1
        ..
26139    1
3222     0
21111    1
42730    1
25326    0
Name: sentiment, Length: 2500, dtype: int64


#**Text to Vector by word2vec**

In [9]:
import multiprocessing
print("CPU cores:", multiprocessing.cpu_count())
CPU_CORES=4

CPU cores: 4


In [10]:
tokenized_train = train_rev.tolist() # pandas Series -> list
vectorize_model = Word2Vec(
    sentences=tokenized_train, 
    vector_size=HIDDEN_DIM, # embedding size
    window=5,
    min_count=1,
    workers=CPU_CORES,
    sg=1 # 0 = CBOW, 1 = Skip-gram
)

print(type(tokenized_train))
print(type(tokenized_train[0]))
print(tokenized_train[0][:10])

<class 'list'>
<class 'list'>
['recently', 'started', 'watching', 'show', 'say', 'really', 'made', 'laugh', 'appreciate', 'unrealistic']


# สร้าง embedding matrix ตาม vocab for initial weight

In [11]:
# text to vector
X_train = train_rev 
X_val = val_rev
X_test = test_rev

y_train = (train_sent).astype(np.int64).to_numpy()
y_val   = (val_sent).astype(np.int64).to_numpy()
y_test  = (test_sent).astype(np.int64).to_numpy()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)
print("=========================")
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

X_train: (45000,)
X_val: (2500,)
X_test: (2500,)
y_train: (45000,)
y_val: (2500,)
y_test: (2500,)


In [12]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [13]:
counter = Counter()
for t in train_rev:
    counter.update(t)

vocab = {PAD_TOKEN: PAD_ID, UNK_TOKEN: UNK_ID}
for i, (w, _) in enumerate(counter.most_common(MAX_VOCAB - 2), start=2):
    vocab[w] = i

def encode(tokens_list):
    ids = [vocab.get(t, vocab[UNK_TOKEN]) for t in tokens_list][:MAX_LEN]
    return ids + [0] * (MAX_LEN - len(ids))

vocab_size = len(vocab) # 

# Create token_id

In [14]:
embedding_weight = np.zeros((vocab_size, HIDDEN_DIM), dtype=np.float32)

# ตารางคำศัพท์ → เวกเตอร์
# PAD row (0) = 0 
# UNK row (1) สุ่มเล็กน้อย
rng = np.random.default_rng(RANDOM_STATE)
embedding_weight[UNK_ID] = rng.normal(0, 0.01, size=(HIDDEN_DIM,)).astype(np.float32)

for word, idx in vocab.items():
    if word in (PAD_TOKEN, UNK_TOKEN):
        continue
    if word in vectorize_model.wv:
        embedding_weight[idx] = vectorize_model.wv[word]

# X-transformer Part

In [15]:
class ImdbDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts.tolist()     # เช่น train_rev
        self.labels = labels         # numpy array y_train (0/1)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ids = encode(self.texts[idx])                 # list length = MAX_LEN
        x = torch.tensor(ids, dtype=torch.long)       # (T,)
        y = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return x, y


In [16]:
class XTransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_matrix, num_classes=2):
        super().__init__()
        # self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        # if embedding_matrix is not None:
        #     self.embedding.weight.data.copy_(torch.tensor(embedding_matrix, dtype=torch.float32))
        # self.embedding.weight.requires_grad = True
        
        self.transformer = TransformerWrapper(
            num_tokens=vocab_size,
            max_seq_len=MAX_LEN,
            attn_layers=Decoder(
                dim=HIDDEN_DIM,
                depth=DEPTH,
                heads=HEADS,
            )
        )

        self.load_w2v_into_wrapper(self.transformer, embedding_weight=embedding_matrix)
        self.emb_dropout = nn.Dropout(0.2)
        self.cls_dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(HIDDEN_DIM, num_classes)
    
    def load_w2v_into_wrapper(self, wrapper, embedding_weight):
        W = torch.tensor(embedding_weight, dtype=torch.float32)
        
        for name, module in wrapper.named_modules():
            if isinstance(module, nn.Embedding) and module.weight.shape == W.shape:
                module.weight.data.copy_(W)
                module.weight.requires_grad = True
                module.weight.data[0].zero_()
                print("✅ Loaded w2v into:", name, module.weight.shape) # expect [vocab_size, embedding_dim]
                return
        
        raise RuntimeError("❌ not found nn.Embedding in TransformerWrapper for this w2v shape")

    def forward(self, input_ids):
        mask = (input_ids != 0)  # (B,T)  padding_idx=0
        emb = self.transformer(input_ids, mask=mask, return_embeddings=True)  # (B,T,D)
        emb = self.emb_dropout(emb)  # dropout บน token representations
        # print(type(emb), emb.shape)
        # print(self.transformer)
        mask = mask.unsqueeze(-1).float()  # (B,T,1)
        pooled = (emb * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)  # (B,D)
        pooled = self.cls_dropout(pooled)
        return self.classifier(pooled)



In [17]:
model = XTransformerClassifier(vocab_size=vocab_size, embedding_matrix=embedding_weight).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

✅ Loaded w2v into: token_emb.emb torch.Size([20000, 300])


In [18]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total = 0.0, 0

    all_preds = []
    all_labels = []

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item() * y.size(0)
        total += y.size(0)

        preds = logits.argmax(dim=1)

        all_preds.append(preds.detach().cpu().numpy())
        all_labels.append(y.detach().cpu().numpy())

    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_labels)

    acc = (y_pred == y_true).mean()
    f1 = f1_score(y_true, y_pred, average="binary") 
    
    return total_loss / total, acc, f1

In [19]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * y.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)

    return running_loss / total, correct / total

In [20]:
def fit(model, train_loader, val_loader, optimizer, criterion, device, epochs):
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_f1": []}

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc, val_f1 = evaluate(model, val_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(
            f"Epoch {epoch:02d} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc*100:.2f}% | "
            f"val_loss={val_loss:.4f} val_acc={val_acc*100:.2f}% val_f1={val_f1:.4f}"
        )

    return history

In [21]:
def test(model, test_loader, criterion, device):
    test_loss, test_acc, test_f1 = evaluate(model, test_loader, criterion, device)
    print(f"TEST | loss={test_loss:.4f} acc={test_acc*100:.2f}% f1={test_f1:.4f}")
    return test_loss, test_acc, test_f1

In [22]:
train_ds = ImdbDataset(X_train, y_train)
val_ds   = ImdbDataset(X_val, y_val)
test_ds  = ImdbDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# Train and Test

In [23]:
history_ = fit(model, train_loader, val_loader, optimizer, criterion, DEVICE, epochs=EPOCHS)
test_loss, test_acc,test_f1 = test(model, test_loader, criterion, DEVICE)

Epoch 01 | train_loss=0.3938 train_acc=81.64% | val_loss=0.2948 val_acc=88.04% val_f1=0.8764
Epoch 02 | train_loss=0.2450 train_acc=90.23% | val_loss=0.2682 val_acc=88.76% val_f1=0.8913
Epoch 03 | train_loss=0.1847 train_acc=92.86% | val_loss=0.3136 val_acc=87.72% val_f1=0.8702
Epoch 04 | train_loss=0.1365 train_acc=94.95% | val_loss=0.3749 val_acc=87.96% val_f1=0.8780
Epoch 05 | train_loss=0.0981 train_acc=96.50% | val_loss=0.3512 val_acc=87.88% val_f1=0.8793
TEST | loss=0.3500 acc=88.12% f1=0.8828


In [24]:
@torch.no_grad()
def predict_text(model, text, device):
    model.eval()

    # preprocess
    text = sw_remove(process(text))
    
    ids = encode(text)
    x = torch.tensor([ids], dtype=torch.long, device=device)  # (1, MAX_LEN)

    logits = model(x)
    probs = torch.softmax(logits, dim=1).squeeze(0)  # (2,)

    pred_id = int(torch.argmax(probs).item())
    confidence = float(probs[pred_id].item()) * 100

    label = "positive" if pred_id == 1 else "negative"
    return label, confidence, probs.detach().cpu().numpy()

In [25]:
label, conf, probs = predict_text(model,"This movie was amazing and fun!5555", DEVICE)
print("result:",label, f"{conf:.8f}%")

result: positive 99.47394729%
